In [ ]:
# PATH CONFIGURATION ONLY: edit these locations for your runtime.
# In Colab, upload the archive into WORK_ROOT, or set DATASET_ZIP explicitly.
# Configure absolute locations before running subsequent cells in order.
from pathlib import Path
import os

WORK_ROOT = Path.cwd().resolve()
DATASET_ZIP = WORK_ROOT / "rgbd_dataset_freiburg1_desk.zip"
DATASET_EXTRACT_ROOT = WORK_ROOT / "fr1_desk_data"
RESULTS_ROOT = WORK_ROOT / "fr1_desk_LightGlue_results"
LIGHTGLUE_DIR = WORK_ROOT / "LightGlue"
RESULT_ARCHIVE_BASE = WORK_ROOT / "fr1_desk_LightGlue_results"
VERIFICATION_ROOT = WORK_ROOT
# Search only the configured sequence extraction tree.
DATASET_SEARCH_ROOT = DATASET_EXTRACT_ROOT
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
VERIFICATION_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# FR1/DESK SUPERPOINT + LIGHTGLUE RAW MATCH EXPORT
# UPDATED VERSION
# ============================================================

# ============================================================
# 1. INSTALL LIGHTGLUE
# ============================================================

!pip -q install git+https://github.com/cvg/LightGlue.git


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import glob
import zipfile
import numpy as np
import pandas as pd
import torch

from PIL import Image
from tqdm.auto import tqdm

from lightglue import LightGlue, SuperPoint
from lightglue.utils import rbd


# ============================================================
# 3. CHECK GPU
# ============================================================

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("\nWARNING:")
    print("GPU is NOT enabled.")
    print("This experiment will be much slower on CPU.")
    print("Use Runtime -> Change runtime type -> GPU\n")


# ============================================================
# 4. PATHS
# ============================================================

ZIP_PATH = str(DATASET_ZIP)

EXTRACT_ROOT = str(DATASET_EXTRACT_ROOT)

OUTPUT_ROOT = str(RESULTS_ROOT)

os.makedirs(
    EXTRACT_ROOT,
    exist_ok=True
)

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True
)


# ============================================================
# 5. EXTRACT DATASET
# ============================================================

if not os.path.isfile(ZIP_PATH):

    raise FileNotFoundError(
        "\nDataset ZIP not found.\n"
        "Upload this file to Colab:\n"
        "rgbd_dataset_freiburg1_desk.zip\n"
        "Expected location:\n"
        f"{ZIP_PATH}"
    )


# Avoid unnecessary re-extraction if already present
existing_pngs = glob.glob(
    os.path.join(
        EXTRACT_ROOT,
        "**",
        "rgb",
        "*.png"
    ),
    recursive=True
)

if len(existing_pngs) == 0:

    print("Extracting dataset...")

    with zipfile.ZipFile(
        ZIP_PATH,
        "r"
    ) as z:

        z.extractall(EXTRACT_ROOT)

    print("Dataset extracted.")

else:

    print("Dataset already extracted.")


# ============================================================
# 6. FIND RGB FOLDER
# ============================================================

rgb_candidates = []

for root, dirs, files in os.walk(EXTRACT_ROOT):

    if os.path.basename(root).lower() == "rgb":

        png_files = glob.glob(
            os.path.join(
                root,
                "*.png"
            )
        )

        if len(png_files) > 0:

            rgb_candidates.append(root)


if len(rgb_candidates) == 0:

    raise RuntimeError(
        "Could not find the RGB folder."
    )


RGB_FOLDER = rgb_candidates[0]

print("\nRGB folder:")
print(RGB_FOLDER)


# ============================================================
# 7. LOAD AND SORT RGB FILES
# ============================================================

def get_timestamp(path):

    filename = os.path.basename(path)

    name, _ = os.path.splitext(filename)

    return float(name)


rgb_files = glob.glob(
    os.path.join(
        RGB_FOLDER,
        "*.png"
    )
)

# Sort numerically by timestamp
rgb_files = sorted(
    rgb_files,
    key=get_timestamp
)


print("\nNumber of RGB images:", len(rgb_files))


if len(rgb_files) != 613:

    print(
        "WARNING: fr1/desk normally contains "
        "613 RGB images."
    )


# ============================================================
# 8. PAIR SETTINGS
# ============================================================

# MATLAB Experiment 20:
#
# startIdx = 100
# numPairs = 50
# frameGap = 3
#
# Python is zero-indexed:
# MATLAB 100 = Python 99

START_INDEX = 99

GAP = 3

N_PAIRS = 50


last_required_index = (
    START_INDEX
    + N_PAIRS
    - 1
    + GAP
)


if last_required_index >= len(rgb_files):

    raise RuntimeError(
        "Not enough images for requested pairs."
    )


print("\nPAIR SETTINGS")
print("--------------------------------")

print(
    "START_INDEX Python:",
    START_INDEX
)

print(
    "Equivalent MATLAB START_INDEX:",
    START_INDEX + 1
)

print(
    "GAP:",
    GAP
)

print(
    "Number of pairs:",
    N_PAIRS
)


# ============================================================
# 9. VERIFY FIRST AND LAST PAIRS
# ============================================================

first_i = START_INDEX
first_j = first_i + GAP

last_i = (
    START_INDEX
    + N_PAIRS
    - 1
)

last_j = last_i + GAP


print("\nPAIR VERIFICATION")
print("--------------------------------")

print("First pair:")
print(
    os.path.basename(
        rgb_files[first_i]
    )
)
print(
    os.path.basename(
        rgb_files[first_j]
    )
)

print("\nLast pair:")
print(
    os.path.basename(
        rgb_files[last_i]
    )
)
print(
    os.path.basename(
        rgb_files[last_j]
    )
)


# ============================================================
# 10. IMAGE SETTINGS
# ============================================================

H = 480
W = 640


# ============================================================
# 11. PWM / ROLLING-SHUTTER SETTINGS
# ============================================================

PWM_FREQ_HZ = 100.0

EXPOSURE_S = 0.001

DUTY_CYCLE = 0.25

MOD_DEPTH = 0.90

NUM_SAMPLES = 100


PHASES = [
    0.0,
    0.125,
    0.25,
    0.5,
    0.875
]


READOUT_MS = [
    5,
    10,
    15,
    20,
    25,
    30
]


N_CONDITIONS = (
    len(PHASES)
    * len(READOUT_MS)
)


print("\nPWM SETTINGS")
print("--------------------------------")

print(
    "PWM frequency:",
    PWM_FREQ_HZ,
    "Hz"
)

print(
    "Exposure:",
    EXPOSURE_S * 1000,
    "ms"
)

print(
    "Duty cycle:",
    DUTY_CYCLE
)

print(
    "Modulation depth:",
    MOD_DEPTH
)

print(
    "Temporal samples:",
    NUM_SAMPLES
)

print(
    "Phases:",
    PHASES
)

print(
    "Readout times:",
    READOUT_MS
)

print(
    "Total conditions:",
    N_CONDITIONS
)


# ============================================================
# 12. PWM ROW-GAIN FUNCTION
# ============================================================

def generate_row_gain(
    height,
    readout_s,
    exposure_s,
    pwm_freq_hz,
    duty_cycle,
    modulation_depth,
    phase_offset
):

    row_time = (
        readout_s
        / height
    )

    exposure_samples = np.linspace(
        0.0,
        exposure_s,
        NUM_SAMPLES
    )

    row_gain = np.zeros(
        height,
        dtype=np.float64
    )

    off_intensity = (
        1.0
        - modulation_depth
    )

    for row in range(height):

        row_start = (
            row
            * row_time
        )

        sample_times = (
            row_start
            + exposure_samples
        )

        phase = np.mod(
            sample_times
            * pwm_freq_hz
            + phase_offset,
            1.0
        )

        led_on = (
            phase
            < duty_cycle
        )

        intensity = np.where(
            led_on,
            1.0,
            off_intensity
        )

        row_gain[row] = np.mean(
            intensity
        )

    return row_gain


# ============================================================
# 13. APPLY ROW GAIN
# ============================================================

def apply_row_gain_rgb(
    image_uint8,
    row_gain
):

    image_float = image_uint8.astype(
        np.float64
    )

    gain = row_gain[
        :,
        None,
        None
    ]

    perturbed = (
        image_float
        * gain
    )

    # Final MATLAB/Python matched convention
    perturbed = np.clip(
        np.rint(perturbed),
        0,
        255
    ).astype(np.uint8)

    return perturbed


# ============================================================
# 14. LOAD SUPERPOINT + LIGHTGLUE
# ============================================================

extractor = SuperPoint(
    max_num_keypoints=2048
).eval().to(device)


matcher = LightGlue(
    features="superpoint"
).eval().to(device)


print(
    "\nSuperPoint + LightGlue loaded."
)


# ============================================================
# 15. NUMPY IMAGE -> TORCH
# ============================================================

def numpy_rgb_to_tensor(img):

    arr = (
        img.astype(np.float32)
        / 255.0
    )

    tensor = torch.from_numpy(
        arr
    ).permute(
        2,
        0,
        1
    )

    return tensor.to(device)


# ============================================================
# 16. UPDATED LIGHTGLUE MATCH FUNCTION
# ============================================================

@torch.inference_mode()
def match_images(
    img1,
    img2
):

    # --------------------------------------------------------
    # Convert to PyTorch
    # --------------------------------------------------------

    t1 = numpy_rgb_to_tensor(
        img1
    )

    t2 = numpy_rgb_to_tensor(
        img2
    )


    # --------------------------------------------------------
    # SuperPoint
    # --------------------------------------------------------

    feats0 = extractor.extract(
        t1,
        resize=None
    )

    feats1 = extractor.extract(
        t2,
        resize=None
    )


    # --------------------------------------------------------
    # LightGlue
    # --------------------------------------------------------

    matches01 = matcher({
        "image0": feats0,
        "image1": feats1
    })


    # --------------------------------------------------------
    # Correct batch removal
    # --------------------------------------------------------

    feats0 = rbd(feats0)

    feats1 = rbd(feats1)

    matches01 = rbd(
        matches01
    )


    # --------------------------------------------------------
    # Get match indices
    # --------------------------------------------------------

    matches = matches01[
        "matches"
    ]


    if (
        matches is None
        or matches.numel() == 0
    ):

        return (
            np.empty(
                (0, 2),
                dtype=np.float32
            ),
            np.empty(
                (0, 2),
                dtype=np.float32
            )
        )


    # --------------------------------------------------------
    # Get keypoints
    # --------------------------------------------------------

    keypoints0 = feats0[
        "keypoints"
    ]

    keypoints1 = feats1[
        "keypoints"
    ]


    matched0 = keypoints0[
        matches[:, 0]
    ]

    matched1 = keypoints1[
        matches[:, 1]
    ]


    p1 = (
        matched0
        .detach()
        .cpu()
        .numpy()
    )

    p2 = (
        matched1
        .detach()
        .cpu()
        .numpy()
    )


    return p1, p2


# ============================================================
# 17. LOAD RGB IMAGE
# ============================================================

def read_rgb(path):

    img = Image.open(
        path
    ).convert(
        "RGB"
    )

    arr = np.array(
        img
    )

    if (
        arr.shape[0] != H
        or arr.shape[1] != W
    ):

        raise RuntimeError(
            f"Unexpected image size: "
            f"{arr.shape} "
            f"for {path}"
        )

    return arr


# ============================================================
# 18. QUICK TEST BEFORE FULL RUN
# ============================================================

print(
    "\n========================================"
)

print(
    "QUICK LIGHTGLUE TEST"
)

print(
    "========================================"
)


test_i = START_INDEX

test_j = (
    test_i
    + GAP
)


test_img1 = read_rgb(
    rgb_files[test_i]
)

test_img2 = read_rgb(
    rgb_files[test_j]
)


test_p1, test_p2 = match_images(
    test_img1,
    test_img2
)


print(
    "Test pair matches:",
    len(test_p1)
)


if len(test_p1) == 0:

    print(
        "WARNING: zero matches "
        "for clean test pair."
    )

else:

    print(
        "Quick test successful."
    )


# ============================================================
# 19. CLEAN LIGHTGLUE EXPORT
# ============================================================

clean_rows = []


print(
    "\n========================================"
)

print(
    "GENERATING CLEAN LIGHTGLUE MATCHES"
)

print(
    "========================================\n"
)


for pair_id in tqdm(
    range(
        1,
        N_PAIRS + 1
    )
):

    i = (
        START_INDEX
        + pair_id
        - 1
    )

    j = (
        i
        + GAP
    )


    file1 = rgb_files[i]

    file2 = rgb_files[j]


    img1 = read_rgb(
        file1
    )

    img2 = read_rgb(
        file2
    )


    p1, p2 = match_images(
        img1,
        img2
    )


    timestamp1 = get_timestamp(
        file1
    )

    timestamp2 = get_timestamp(
        file2
    )


    for match_id in range(
        len(p1)
    ):

        clean_rows.append({

            "PairID":
                pair_id,

            "ImageIndex1":
                i + 1,

            "ImageIndex2":
                j + 1,

            "Timestamp1":
                timestamp1,

            "Timestamp2":
                timestamp2,

            "MatchID":
                match_id + 1,

            "X1":
                float(
                    p1[
                        match_id,
                        0
                    ]
                ),

            "Y1":
                float(
                    p1[
                        match_id,
                        1
                    ]
                ),

            "X2":
                float(
                    p2[
                        match_id,
                        0
                    ]
                ),

            "Y2":
                float(
                    p2[
                        match_id,
                        1
                    ]
                )

        })


    if pair_id % 10 == 0:

        print(
            f"Clean pair "
            f"{pair_id}/"
            f"{N_PAIRS} | "
            f"matches = "
            f"{len(p1)}"
        )


# ============================================================
# 20. SAVE CLEAN CSV
# ============================================================

clean_df = pd.DataFrame(
    clean_rows
)


clean_file = os.path.join(
    OUTPUT_ROOT,
    "fr1_desk_LightGlue_clean_matches.csv"
)


clean_df.to_csv(
    clean_file,
    index=False
)


print(
    "\nClean CSV saved:"
)

print(
    clean_file
)

print(
    "Total clean raw matches:",
    len(clean_df)
)


# ============================================================
# 21. PWM LIGHTGLUE EXPORT
# ============================================================

pwm_rows = []


total_evaluations = (
    N_PAIRS
    * len(READOUT_MS)
    * len(PHASES)
)


evaluation_counter = 0


print(
    "\n========================================"
)

print(
    "GENERATING PWM LIGHTGLUE MATCHES"
)

print(
    "========================================"
)

print(
    "Total pair-condition evaluations:",
    total_evaluations
)

print()


for pair_id in range(
    1,
    N_PAIRS + 1
):

    i = (
        START_INDEX
        + pair_id
        - 1
    )

    j = (
        i
        + GAP
    )


    file1 = rgb_files[i]

    file2 = rgb_files[j]


    clean_img1 = read_rgb(
        file1
    )

    clean_img2 = read_rgb(
        file2
    )


    timestamp1 = get_timestamp(
        file1
    )

    timestamp2 = get_timestamp(
        file2
    )


    for readout_ms in READOUT_MS:

        readout_s = (
            readout_ms
            / 1000.0
        )


        # Frame 1 always phase 0
        row_gain1 = generate_row_gain(
            H,
            readout_s,
            EXPOSURE_S,
            PWM_FREQ_HZ,
            DUTY_CYCLE,
            MOD_DEPTH,
            0.0
        )


        img1_pwm = apply_row_gain_rgb(
            clean_img1,
            row_gain1
        )


        for phase2 in PHASES:

            row_gain2 = generate_row_gain(
                H,
                readout_s,
                EXPOSURE_S,
                PWM_FREQ_HZ,
                DUTY_CYCLE,
                MOD_DEPTH,
                phase2
            )


            img2_pwm = apply_row_gain_rgb(
                clean_img2,
                row_gain2
            )


            p1, p2 = match_images(
                img1_pwm,
                img2_pwm
            )


            evaluation_counter += 1


            for match_id in range(
                len(p1)
            ):

                pwm_rows.append({

                    "PairID":
                        pair_id,

                    "ImageIndex1":
                        i + 1,

                    "ImageIndex2":
                        j + 1,

                    "Timestamp1":
                        timestamp1,

                    "Timestamp2":
                        timestamp2,

                    "Phase":
                        float(
                            phase2
                        ),

                    "ReadoutMs":
                        float(
                            readout_ms
                        ),

                    "MatchID":
                        match_id + 1,

                    "X1":
                        float(
                            p1[
                                match_id,
                                0
                            ]
                        ),

                    "Y1":
                        float(
                            p1[
                                match_id,
                                1
                            ]
                        ),

                    "X2":
                        float(
                            p2[
                                match_id,
                                0
                            ]
                        ),

                    "Y2":
                        float(
                            p2[
                                match_id,
                                1
                            ]
                        )

                })


            if (
                evaluation_counter
                % 25
                == 0
            ):

                print(
                    f"{evaluation_counter}/"
                    f"{total_evaluations} "
                    f"evaluations complete"
                )


# ============================================================
# 22. SAVE PWM CSV
# ============================================================

pwm_df = pd.DataFrame(
    pwm_rows
)


pwm_file = os.path.join(
    OUTPUT_ROOT,
    "fr1_desk_LightGlue_PWM_matches.csv"
)


pwm_df.to_csv(
    pwm_file,
    index=False
)


print(
    "\nPWM CSV saved:"
)

print(
    pwm_file
)

print(
    "Total PWM raw matches:",
    len(pwm_df)
)


# ============================================================
# 23. FINAL VERIFICATION
# ============================================================

print(
    "\n========================================"
)

print(
    "FR1/DESK LIGHTGLUE EXPORT COMPLETE"
)

print(
    "========================================"
)


print(
    "\nClean CSV:"
)

print(
    clean_file
)


print(
    "\nPWM CSV:"
)

print(
    pwm_file
)


print(
    "\nClean rows:",
    len(clean_df)
)


print(
    "PWM rows:",
    len(pwm_df)
)


print(
    "Unique clean pairs:",
    clean_df[
        "PairID"
    ].nunique()
)


print(
    "Unique PWM pairs:",
    pwm_df[
        "PairID"
    ].nunique()
)


print(
    "Unique phases:",
    sorted(
        pwm_df[
            "Phase"
        ].unique()
    )
)


print(
    "Unique readout times:",
    sorted(
        pwm_df[
            "ReadoutMs"
        ].unique()
    )
)


print(
    "Expected pair-condition evaluations:",
    total_evaluations
)


# ============================================================
# 24. ADDITIONAL CONSISTENCY CHECK
# ============================================================

condition_counts = (
    pwm_df[
        [
            "PairID",
            "Phase",
            "ReadoutMs"
        ]
    ]
    .drop_duplicates()
)


print(
    "\nUnique pair-condition combinations:",
    len(condition_counts)
)


if len(condition_counts) == 1500:

    print(
        "PASS: all 1500 "
        "pair-condition combinations present."
    )

else:

    print(
        "WARNING: expected 1500 "
        "pair-condition combinations."
    )


print(
    "\n========================================"
)

print(
    "DONE"
)

print(
    "========================================"
)